In [ ]:
import os
import calendar
import time
import random
import zipfile
from pathlib import Path
from copy import deepcopy

import cdsapi
from tqdm.auto import tqdm


In [ ]:
# Point cdsapi to the .cdsapirc that lives INSIDE your repo
# (cdsapi otherwise expects it in $HOME/.cdsapirc) 
rc_path = Path(".") / ".cdsapirc"
assert rc_path.exists(), f"Repo .cdsapirc not found: {rc_path.resolve()}"

os.environ["CDSAPI_RC"] = str(rc_path.resolve())

# Optional safety: ensure you’re using EWDS endpoint (matches EWDS docs)
os.environ["CDSAPI_URL"] = "https://ewds.climate.copernicus.eu/api"  

client = cdsapi.Client()

In [ ]:
dataset = "cems-glofas-historical"

SYSTEM_VERSION = "version_4_0"
PRODUCT_TYPE = "consolidated"
VARIABLE = "river_discharge_in_the_last_24_hours"
HYDRO_MODEL = "lisflood"

EXTRACT = True  # set True if you want extraction

# whole-country bbox in [N, W, S, E] 
AREA = [35, 63, 4, 131]

START_YEAR = 1999 #1979
END_YEAR   = 2023 #2025  # inclusive

months = [f"{m:02d}" for m in range(1, 13)]
days   = [f"{d:02d}" for d in range(1, 32)]

OUT_DIR = (
    Path("data/raw/glofas/historical")
    / SYSTEM_VERSION
    / PRODUCT_TYPE
    / "discharge"
    / "grib2"
    / f"area_{AREA[0]}_{AREA[1]}_{AREA[2]}_{AREA[3]}"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

def target_zip_for_year(year: int) -> Path:
    # Match your naming convention + add .zip extension
    # Example you gave: glofas_historical_version_4_0_consolidated_river_discharge_in_the_last_24_hours_2025
    # We'll store as ..._2025.zip (more explicit / safer).
    stem = f"glofas_historical_{SYSTEM_VERSION}_{PRODUCT_TYPE}_{VARIABLE}_{year}"
    return OUT_DIR / f"{stem}.zip"

print("Output folder:", OUT_DIR)
print("Example target:", target_zip_for_year(2025))


In [ ]:
BASE_REQUEST = {
    "system_version": [SYSTEM_VERSION],
    "hydrological_model": [HYDRO_MODEL],
    "product_type": [PRODUCT_TYPE],
    "variable": [VARIABLE],
    "hmonth": months,
    "hday": days,
    "data_format": "grib2",
    "download_format": "zip",
    "area": AREA,
}

def is_valid_zip(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    # quick structural check
    if not zipfile.is_zipfile(path):
        return False
    # deeper check (optional but good): try listing contents
    try:
        with zipfile.ZipFile(path, "r") as zf:
            _ = zf.namelist()[:5]
        return True
    except Exception:
        return False

def retrieve_with_retries(dataset: str, request: dict, target: Path, max_attempts: int = 5):
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            # 3rd argument writes directly to your target path :contentReference[oaicite:6]{index=6}
            client.retrieve(dataset, request, str(target))
            return
        except Exception as e:
            last_err = e
            # Backoff with jitter
            sleep_s = min(120, 5 * attempt) + random.uniform(0, 2)
            print(f"[Attempt {attempt}/{max_attempts}] Error: {e}\nSleeping {sleep_s:.1f}s then retrying...")
            time.sleep(sleep_s)
    raise last_err



In [ ]:
years = list(range(START_YEAR, END_YEAR + 1))

for year in tqdm(years, desc="Downloading yearly ZIPs"):
    target = target_zip_for_year(year)

    # Support BOTH possibilities:
    # - our preferred target "..._YYYY.zip"
    # - your existing files without ".zip" suffix
    legacy_target = Path(str(target)[:-4])  # removes ".zip"
    have_file = is_valid_zip(target) or is_valid_zip(legacy_target)

    if have_file:
        tqdm.write(f"✓ {year} already present (skipping)")
        continue

    # If a previous partial download exists, remove it
    for p in [target, legacy_target]:
        if p.exists() and not is_valid_zip(p):
            tqdm.write(f"⚠ Removing corrupt/partial file: {p.name}")
            try:
                p.unlink()
            except Exception:
                pass

    req = deepcopy(BASE_REQUEST)
    req["hyear"] = [str(year)]

    tqdm.write(f"↓ Downloading {year} → {target.name}")
    retrieve_with_retries(dataset, req, target, max_attempts=5)

    if not is_valid_zip(target):
        raise RuntimeError(f"Downloaded file is not a valid ZIP: {target}")

    tqdm.write(f"✓ Done {year} (size={target.stat().st_size/1e6:.1f} MB)")
    if EXTRACT:
        extract_dir = OUT_DIR / str(year)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target, "r") as zf:
            zf.extractall(extract_dir)
        tqdm.write(f"↳ Extracted to {extract_dir}")


In [1]:
import os
import json
import time
import random
import calendar
import zipfile
import shutil
import re
from pathlib import Path
from datetime import date

import requests
from tqdm.std import tqdm
from concurrent.futures import ThreadPoolExecutor

from ecmwf.datastores import Client as DSClient

# ==========================
# EWDS reforecast schedule helpers
# ==========================
FIRST_REF_DATE = date(2003, 3, 27)
REF_WEEKDAYS = (0, 3)  # Mon/Thu

def tlog(msg: str) -> None:
    try:
        tqdm.write(msg)
    except Exception:
        print(msg, flush=True)

def ref_year_for_hyear(hyear: int) -> int:
    return hyear + 20

def valid_hdays_for_month(hyear: int, month: int) -> list[str]:
    ry = ref_year_for_hyear(hyear)
    ndays = calendar.monthrange(ry, month)[1]
    out = []
    for d in range(1, ndays + 1):
        ref_dt = date(ry, month, d)
        if ref_dt < FIRST_REF_DATE:
            continue
        if ref_dt.weekday() in REF_WEEKDAYS:
            out.append(f"{d:02d}")
    return out

# ==========================
# User config
# ==========================
RAW_ROOT = Path(r"G:\My Drive\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\reforecast\version_4_0\consolidated\discharge\grib2\area_35_63_4_131")
DATASET = "cems-glofas-reforecast"
SYSTEM_VERSION = "version_4_0"

AREA = [35, 63, 4, 131]  # [N, W, S, E]
LEADTIMES = [str(x) for x in range(24, 1104 + 1, 24)]

HYEARS = list(range(2012, 2023 + 1)) #2006 in process
FORCE = False

# Disk-space controls
KEEP_ZIPS = False                 # always delete zips ASAP
KEEP_JOBS_STATE_AFTER_YEAR = True # keep jobs_state.json for audit/resume

# Concurrency / polling controls
DOWNLOAD_WORKERS = 3      # lower = fewer simultaneous ZIPs on disk
MAX_INFLIGHT = 12         # lower = fewer jobs at once
POLL_SECONDS = 30
JITTER_SECONDS = 5
MAX_SUBMIT_RETRIES = 3
MAX_DOWNLOAD_RETRIES = 3

# Monthly output normalization
MONTHLY_OUT_NAME = "data.grib"    # each month folder will end with this single file

# Point to repo-local .cdsapirc
rc_path = Path(".") / ".cdsapirc"
assert rc_path.exists(), f"Repo .cdsapirc not found: {rc_path.resolve()}"
os.environ["CDSAPI_RC"] = str(rc_path.resolve())
os.environ["CDSAPI_URL"] = "https://ewds.climate.copernicus.eu/api"

def parse_cdsapirc(path: Path) -> tuple[str, str]:
    url = None
    key = None
    for line in path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if s.startswith("url:"):
            url = s.split(":", 1)[1].strip()
        if s.startswith("key:"):
            key = s.split(":", 1)[1].strip()
    if not url or not key:
        raise RuntimeError(f"Could not parse url/key from {path}")
    token = key.split(":", 1)[1] if ":" in key else key
    return url, token

EWDS_URL, EWDS_TOKEN = parse_cdsapirc(rc_path)

# Single client for submit + polling (main thread)
ds = DSClient(url=EWDS_URL, key=EWDS_TOKEN)

# ==========================
# IO helpers
# ==========================
def mb(p: Path) -> float:
    return p.stat().st_size / (1024 * 1024)

def safe_unlink(path: Path) -> None:
    try:
        if path.exists():
            path.unlink()
    except Exception:
        pass

def safe_rmtree(path: Path) -> None:
    try:
        if path.exists():
            shutil.rmtree(path, ignore_errors=True)
    except Exception:
        pass

def is_valid_zip(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    if not zipfile.is_zipfile(path):
        return False
    try:
        with zipfile.ZipFile(path, "r") as zf:
            _ = zf.namelist()[:5]
        return True
    except Exception:
        return False

def is_grib_payload(p: Path) -> bool:
    """True GRIB payload: not .idx and starts with magic bytes 'GRIB'."""
    if (not p.is_file()) or p.stat().st_size == 0:
        return False
    if p.name.lower().endswith(".idx"):
        return False
    try:
        with open(p, "rb") as f:
            return f.read(4) == b"GRIB"
    except Exception:
        return False

def has_monthly_data_grib(month_dir: Path) -> bool:
    p = month_dir / MONTHLY_OUT_NAME
    return is_grib_payload(p)

def extract_zip(zip_path: Path, extract_dir: Path) -> list[Path]:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    # Return GRIB-like files (exclude .idx)
    out = []
    for p in extract_dir.rglob("*.grib*"):
        if p.is_file() and (not p.name.lower().endswith(".idx")) and p.stat().st_size > 0:
            out.append(p)
    return sorted(out)

def normalize_month_folder(month_dir: Path) -> Path:
    """
    Ensure month_dir contains exactly one GRIB payload named data.grib.
    - If already has data.grib (valid), do nothing.
    - Else if exactly one GRIB payload exists, rename/move to data.grib.
    - Else if multiple GRIB payloads exist, concatenate them in-place into data.grib (binary stream),
      then delete the parts to save space.
    """
    month_dir.mkdir(parents=True, exist_ok=True)
    target = month_dir / MONTHLY_OUT_NAME

    if is_grib_payload(target):
        # clean stray idx files if any
        for idx in month_dir.rglob("*.idx"):
            safe_unlink(idx)
        return target

    # collect payload candidates (exclude .idx)
    payloads = [p for p in month_dir.rglob("*.grib*") if is_grib_payload(p)]
    if not payloads:
        raise RuntimeError(f"No GRIB payloads found after extraction in: {month_dir}")

    if len(payloads) == 1:
        src = payloads[0]
        if src.resolve() != target.resolve():
            if target.exists():
                safe_unlink(target)
            src.replace(target)
        # remove stray idx files
        for idx in month_dir.rglob("*.idx"):
            safe_unlink(idx)
        return target

    # Multiple payloads → concatenate (fast, low memory) into target, then delete parts.
    payloads = sorted(payloads)  # stable order; if filenames encode dates, this is usually ok
    tmp = month_dir / (MONTHLY_OUT_NAME + ".tmp")

    buf = 128 * 1024 * 1024
    with open(tmp, "wb") as w:
        for part in payloads:
            with open(part, "rb") as r:
                shutil.copyfileobj(r, w, length=buf)

    if target.exists():
        safe_unlink(target)
    tmp.replace(target)

    # delete original parts if they are not the target
    for part in payloads:
        if part.exists() and part.resolve() != target.resolve():
            safe_unlink(part)

    # delete any .idx
    for idx in month_dir.rglob("*.idx"):
        safe_unlink(idx)

    # final sanity
    if not is_grib_payload(target):
        raise RuntimeError(f"Month normalization produced invalid data.grib: {target}")

    return target

def is_job_not_found_error(e: Exception) -> bool:
    if isinstance(e, requests.HTTPError):
        resp = getattr(e, "response", None)
        if resp is not None and getattr(resp, "status_code", None) == 404:
            return True
    msg = str(e).lower()
    return ("404" in msg) and ("job not found" in msg or "deleted" in msg)

# --------------------------
# Ask EWDS for valid hday values (prevents 400 invalid combination)
# --------------------------
_valid_hday_cache: dict[tuple[str, str], list[str]] = {}

def is_bad_request_400(e: Exception) -> bool:
    msg = str(e).lower()
    if "400" in msg and ("bad request" in msg or "invalid request" in msg):
        return True
    if isinstance(e, requests.HTTPError):
        resp = getattr(e, "response", None)
        if resp is not None and getattr(resp, "status_code", None) == 400:
            return True
    return False

def get_valid_hdays_from_api(hyear_str: str, hmonth_str: str, force_refresh: bool = False) -> list[str]:
    key = (hyear_str, hmonth_str)
    if (not force_refresh) and key in _valid_hday_cache:
        return _valid_hday_cache[key]

    probe_req = {
        "system_version": [SYSTEM_VERSION],
        "hydrological_model": ["lisflood"],
        "product_type": ["ensemble_perturbed_reforecast"],
        "variable": ["river_discharge_in_the_last_24_hours"],
        "hyear": [hyear_str],
        "hmonth": [hmonth_str],
        "leadtime_hour": ["24"],
        "data_format": "grib",
        "download_format": "zip",
        "area": AREA,
    }

    constrained = ds.apply_constraints(DATASET, probe_req)

    days = constrained.get("hday") or constrained.get("day") or []
    out = []
    for d in days:
        s = str(d)
        if s.isdigit():
            out.append(f"{int(s):02d}")
        else:
            out.append(s)

    _valid_hday_cache[key] = out
    return out

# ==========================
# Worker: download + unzip + normalize month (thread-safe via local DSClient)
# ==========================
def download_and_extract_worker(tag: str, request_id: str, zip_path: Path, month_dir: Path) -> bool:
    """
    Downloads ZIP to zip_path (temporary), extracts to month_dir, normalizes to month_dir/data.grib,
    deletes ZIP immediately (KEEP_ZIPS=False), and removes any stray .idx.
    """
    # If month already ready, ensure zip is gone, done
    if has_monthly_data_grib(month_dir):
        if zip_path.exists() and not KEEP_ZIPS:
            safe_unlink(zip_path)
        return True

    # remove corrupt partial zip
    if zip_path.exists() and not is_valid_zip(zip_path):
        safe_unlink(zip_path)

    local_ds = DSClient(url=EWDS_URL, key=EWDS_TOKEN)

    for attempt in range(1, MAX_DOWNLOAD_RETRIES + 1):
        try:
            remote = local_ds.get_remote(request_id)
            remote.download(str(zip_path))

            if not is_valid_zip(zip_path):
                raise RuntimeError("downloaded file is not a valid zip")

            _ = extract_zip(zip_path, month_dir)

            # Delete zip ASAP
            if not KEEP_ZIPS:
                safe_unlink(zip_path)

            # Normalize month folder to single data.grib
            normalize_month_folder(month_dir)

            return True

        except Exception as e:
            safe_unlink(zip_path)
            sleep_s = min(120, 10 * attempt) + random.uniform(0, 3)
            tqdm.write(f"⚠ {tag}: download/extract attempt {attempt} failed: {e} (sleep {sleep_s:.1f}s)")
            time.sleep(sleep_s)

    return False

# ==========================
# Async year runner (jobs inflight + parallel downloads)
# ==========================
def download_reforecast_year_async(hyear: int) -> Path:
    """
    Downloads all months for a given hyear into:
      RAW_ROOT/YYYY/_staging/extract/YYYY_MM/data.grib

    Does NOT build yearly data.grib.
    Returns the extract_base path.
    """
    hyear_str = str(hyear)
    year_dir = RAW_ROOT / hyear_str

    staging = year_dir / "_staging"
    zips_dir = staging / "zips"
    extract_base = staging / "extract"
    state_path = staging / "jobs_state.json"

    zips_dir.mkdir(parents=True, exist_ok=True)
    extract_base.mkdir(parents=True, exist_ok=True)

    if state_path.exists():
        state = json.loads(state_path.read_text(encoding="utf-8"))
    else:
        state = {"months": {}}

    def save_state():
        state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")

    # Build tasks (one per month)
    tasks = []
    for m in range(1, 13):
        days = get_valid_hdays_from_api(hyear_str, f"{m:02d}")
        if not days:
            continue
        tag = f"{hyear_str}_{m:02d}"
        zip_path = zips_dir / f"{tag}.zip"
        month_dir = extract_base / tag  # <-- month folder: YYYY_MM

        tasks.append({
            "tag": tag,
            "month": m,
            "days": days,
            "zip_path": zip_path,
            "month_dir": month_dir,
            "request": {
                "system_version": [SYSTEM_VERSION],
                "hydrological_model": ["lisflood"],
                "product_type": ["ensemble_perturbed_reforecast"],
                "variable": ["river_discharge_in_the_last_24_hours"],
                "hyear": [hyear_str],
                "hmonth": [f"{m:02d}"],
                "hday": days,
                "leadtime_hour": LEADTIMES,
                "data_format": "grib2",
                "download_format": "zip",
                "area": AREA,
            }
        })

    # Resume-friendly DONE detection
    for t in tasks:
        tag = t["tag"]
        if tag not in state["months"]:
            state["months"][tag] = {"status": "pending", "attempts": 0, "request_id": None}

        rec = state["months"][tag]

        # If month already has a valid data.grib -> done
        if has_monthly_data_grib(t["month_dir"]):
            rec["status"] = "done"
            # zip not needed
            if t["zip_path"].exists() and not KEEP_ZIPS:
                safe_unlink(t["zip_path"])

        # If zip exists (valid) but month not normalized, try extract+normalize now
        elif is_valid_zip(t["zip_path"]):
            extract_zip(t["zip_path"], t["month_dir"])
            if not KEEP_ZIPS:
                safe_unlink(t["zip_path"])
            normalize_month_folder(t["month_dir"])
            rec["status"] = "done"

        elif rec.get("status") == "done":
            rec["status"] = "pending"

        # If we crashed mid-download previously, treat as pending again
        if rec.get("status") in ("downloading", "ready") and not rec.get("request_id"):
            rec["status"] = "pending"

    save_state()

    total = len(tasks)
    done = sum(1 for t in tasks if state["months"][t["tag"]]["status"] == "done")
    tqdm.write(f"=== {hyear_str}: {done}/{total} months already done ===")

    # inflight jobs on server
    inflight: dict[str, str] = {}
    for t in tasks:
        tag = t["tag"]
        rec = state["months"][tag]
        rid = rec.get("request_id")
        if rid and rec.get("status") in ("submitted", "running", "ready", "downloading"):
            inflight[tag] = rid

    pbar = tqdm(total=total, initial=done, desc=f"{hyear_str} months", leave=True)

    def submit_task(t):
        tag = t["tag"]
        rec = state["months"][tag]
        req = t["request"]

        for attempt in range(1, MAX_SUBMIT_RETRIES + 1):
            try:
                remote = ds.submit(DATASET, req)
                rec["request_id"] = remote.request_id
                rec["status"] = "submitted"
                rec["attempts"] = rec.get("attempts", 0) + 1
                rec["submitted_at"] = time.time()
                rec["last_error"] = None
                save_state()
                tqdm.write(f"↥ submit {tag} (days={len(t['days'])}) -> {remote.request_id}")
                return remote.request_id
            except Exception as e:
                if is_bad_request_400(e):
                    month_str = t["request"]["hmonth"][0]
                    new_days = get_valid_hdays_from_api(hyear_str, month_str, force_refresh=True)
                    if new_days:
                        t["days"] = new_days
                        t["request"]["hday"] = new_days
                        tqdm.write(f"↻ {tag}: refreshed hday from EWDS constraints (n={len(new_days)}) and retrying submit")
                        continue
                    else:
                        raise RuntimeError(f"{tag}: EWDS constraints returned no valid hday values; cannot submit")

                sleep_s = min(60, 5 * attempt) + random.uniform(0, 2)
                tqdm.write(f"⚠ submit failed {tag} attempt {attempt}: {e} (sleep {sleep_s:.1f}s)")
                time.sleep(sleep_s)

        raise RuntimeError(f"Submit failed permanently for {tag}")

    def reconcile_job_deleted(t):
        tag = t["tag"]
        rec = state["months"][tag]
        zip_path = t["zip_path"]
        month_dir = t["month_dir"]

        if has_monthly_data_grib(month_dir):
            rec["status"] = "done"
            rec["last_error"] = "job_deleted_but_month_present"
            save_state()
            tqdm.write(f"✓ {tag}: job deleted but month data.grib exists -> marked done")
            if zip_path.exists() and not KEEP_ZIPS:
                safe_unlink(zip_path)
            return "done"

        if is_valid_zip(zip_path):
            extract_zip(zip_path, month_dir)
            if not KEEP_ZIPS:
                safe_unlink(zip_path)
            normalize_month_folder(month_dir)
            rec["status"] = "done"
            rec["last_error"] = "job_deleted_but_zip_present_extracted"
            save_state()
            tqdm.write(f"✓ {tag}: job deleted but ZIP exists -> extracted+normalized+done")
            return "done"

        if zip_path.exists() and not is_valid_zip(zip_path):
            safe_unlink(zip_path)

        rec["status"] = "pending"
        rec["request_id"] = None
        rec["last_error"] = "job_deleted_resubmit"
        save_state()
        tqdm.write(f"↻ {tag}: job deleted and no month/zip -> will resubmit")
        return "resubmit"

    # Download pool for this year
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
        download_futures: dict[str, object] = {}
        ready_queue: list[str] = []

        def schedule_download_for_tag(tag: str) -> bool:
            if tag in download_futures:
                return False

            t = next(tt for tt in tasks if tt["tag"] == tag)
            rec = state["months"][tag]
            rid = rec.get("request_id")
            if not rid:
                return False

            if len(download_futures) >= DOWNLOAD_WORKERS:
                return False

            rec["status"] = "downloading"
            save_state()

            fut = executor.submit(
                download_and_extract_worker,
                tag, rid, t["zip_path"], t["month_dir"]
            )
            download_futures[tag] = fut
            tqdm.write(f"⇣ scheduled {tag} download (active={len(download_futures)}/{DOWNLOAD_WORKERS})")
            return True

        while True:
            # 1) Harvest completed downloads
            finished_tags = [tag for tag, fut in download_futures.items() if fut.done()]
            for tag in finished_tags:
                fut = download_futures.pop(tag)
                ok = False
                try:
                    ok = fut.result()
                except Exception as e:
                    tqdm.write(f"⚠ {tag}: worker crashed: {e}")
                    ok = False

                rec = state["months"][tag]
                if ok:
                    rec["status"] = "done"
                    save_state()
                    pbar.update(1)
                    tqdm.write(f"✓ {tag}: download+extract+normalize complete (ZIP deleted)")
                else:
                    rec["status"] = "ready"
                    save_state()
                    if tag not in ready_queue:
                        ready_queue.append(tag)
                    tqdm.write(f"↻ {tag}: download failed -> queued for retry")

            # 2) Progress + termination
            done = sum(1 for t in tasks if state["months"][t["tag"]]["status"] == "done")
            pbar.n = done
            pbar.refresh()
            pbar.set_postfix({"inflight": len(inflight), "dl": len(download_futures), "queue": len(ready_queue)})

            if done >= total and not download_futures:
                break

            progressed = False

            # 3) Schedule queued downloads
            while ready_queue and len(download_futures) < DOWNLOAD_WORKERS:
                tag = ready_queue.pop(0)
                if state["months"][tag]["status"] == "done":
                    continue
                if schedule_download_for_tag(tag):
                    progressed = True

            # 4) Fill inflight jobs
            while len(inflight) < MAX_INFLIGHT:
                next_task = None
                for t in tasks:
                    tag = t["tag"]
                    st = state["months"][tag]["status"]
                    if st == "done":
                        continue
                    if tag in inflight:
                        continue
                    if tag in download_futures or tag in ready_queue:
                        continue
                    if st in ("submitted", "running", "ready", "downloading"):
                        continue
                    next_task = t
                    break

                if next_task is None:
                    break

                rid = submit_task(next_task)
                inflight[next_task["tag"]] = rid
                progressed = True

            # 5) Poll inflight jobs
            for t in tasks:
                tag = t["tag"]
                if tag not in inflight:
                    continue
                if state["months"][tag]["status"] == "done":
                    inflight.pop(tag, None)
                    continue
                if tag in download_futures:
                    inflight.pop(tag, None)
                    continue

                rid = inflight[tag]
                rec = state["months"][tag]

                try:
                    remote = ds.get_remote(rid)
                except Exception as e:
                    if is_job_not_found_error(e):
                        rec["last_error"] = str(e)[:300]
                        save_state()
                        action = reconcile_job_deleted(t)
                        inflight.pop(tag, None)
                        progressed = True
                        if action == "done":
                            pbar.update(1)
                        continue

                    tqdm.write(f"⚠ poll failed {tag}: {e}")
                    rec["last_error"] = str(e)[:300]
                    save_state()
                    continue

                rec["last_poll"] = time.time()
                rec["remote_status"] = getattr(remote, "status", None)
                save_state()

                status = getattr(remote, "status", None)
                ready = getattr(remote, "results_ready", False)

                if status in ("successful", "success") and ready:
                    rec["status"] = "ready"
                    save_state()
                    inflight.pop(tag, None)

                    if not schedule_download_for_tag(tag):
                        if tag not in ready_queue:
                            ready_queue.append(tag)
                            tlog(f"⏸ queued {tag} (no download slot)")
                    progressed = True

                elif status in ("failed", "dismissed", "deleted"):
                    tqdm.write(f"✗ {tag} status={status} -> retry")
                    rec["status"] = "pending"
                    rec["request_id"] = None
                    save_state()
                    inflight.pop(tag, None)
                    progressed = True
                else:
                    rec["status"] = "running"
                    save_state()

            if not progressed:
                time.sleep(POLL_SECONDS + random.uniform(0, JITTER_SECONDS))

    pbar.close()

    # Year-level cleanup: delete ZIP folder (months are kept!)
    # Keep extract_base (contains YYYY_MM/data.grib).
    safe_rmtree(zips_dir)

    if KEEP_JOBS_STATE_AFTER_YEAR:
        # keep only jobs_state.json + extract folder
        staging.mkdir(parents=True, exist_ok=True)
    else:
        # optionally remove jobs_state too
        safe_unlink(state_path)

    tqdm.write(f"🏁 {hyear_str}: completed monthly downloads. Kept: {extract_base} | Deleted ZIPs: {zips_dir}")
    return extract_base

# ==========================
# Run
# ==========================
RAW_ROOT.mkdir(parents=True, exist_ok=True)

for y in tqdm(HYEARS, desc="Years"):
    download_reforecast_year_async(y)

print("DONE.")

Years:   0%|          | 0/12 [00:13<?, ?it/s]

=== 2012: 10/12 months already done ===


                                             
Years:   0%|          | 0/12 [00:14<?, ?it/s]                                  

⇣ scheduled 2012_12 download (active=1/3)


ed455c57a231ad0cf3eef9c2a9777da3.zip:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

                                             
Years:   0%|          | 0/12 [49:08<?, ?it/s]                                            

✓ 2012_12: download+extract+normalize complete (ZIP deleted)


Years:   0%|          | 0/12 [3:30:24<?, ?it/s]


KeyboardInterrupt: 